In [1]:
import json
import math
import os
from collections import defaultdict
from pathlib import Path

import pandas as pd
from datasets import Dataset
from tqdm.auto import tqdm

## Loading the original datset

In [2]:
dataset_path = Path("./formatted-queries_2024-Jan-30_23-29-10.json")
assert dataset_path.exists(), FileNotFoundError("Dataset file does not exist!")

with open(dataset_path, "r", encoding="utf-8") as file:
    dataset = json.load(file)

In [3]:
dataset[0]

['Move robot TCP to coordinates (0.5, 0.3, 0.7) in meters',
 {'functions': [{'function_name': 'move_tcp',
    'inputs': [{'name': 'x', 'value': 0.5, 'unit': 'm'},
     {'name': 'y', 'value': 0.3, 'unit': 'm'},
     {'name': 'z', 'value': 0.7, 'unit': 'm'}]}]}]

## Dataset Reformatting

In [4]:
def convert_angle(value: int | float, unit: str) -> float:
    if unit == "rad":
        return value
    elif unit == "deg":
        return math.radians(value)
    else:
        raise ValueError(f"Unsupported unit in angle: {unit}")


def format_angle(value: int | float | list, unit: str | list):
    if isinstance(unit, list):
        assert len(unit) == len(value), ValueError(
            f"There is not enough units for all values {(unit, value)}"
        )
        return [convert_angle(value=value[i], unit=unit[i]) for i in range(len(unit))]
    elif isinstance(unit, str):
        if isinstance(value, list):
            return [convert_angle(value=value[i], unit=unit) for i in range(len(value))]
        else:
            return convert_angle(value=value, unit=unit)
    else:
        raise ValueError(f"There is an unpredicted unit value in angle: {unit}")


def format_disntace(value: int | float, unit: str):
    if unit == "m":
        return value * 1000
    elif unit == "cm":
        return value * 100
    elif unit == "dm":
        return value * 10
    elif unit == "mm":
        return value
    elif unit == "micrometers":
        return value * 0.001
    else:
        raise ValueError(f"There is an unpredicted unit value in distance: {unit}")

In [5]:
def format_value_new(function_name: str, inputs: list):
    arg_name = inputs["name"]
    value = inputs["value"]
    unit = inputs["unit"]
    if function_name == "move_tcp":
        if isinstance(unit, str):
            return arg_name, format_disntace(value=value, unit=unit)
        elif "q" in arg_name:
            return arg_name, value
        else:
            raise ValueError(f"Unsupported unit in TCP: {unit}")
    elif function_name == "move_joint":
        if arg_name == "joint":
            return arg_name, value if isinstance(value, list) else [value]

        elif arg_name == "angle":
            return arg_name, format_angle(value=value, unit=unit)

        else:
            raise ValueError(
                f"There is no argument name {arg_name} in function: {function_name}"
            )

    # elif function_name == "get_joint_values":

    else:
        raise ValueError(f"There is no function with name: {function_name} in dataset!")

In [6]:
outputs = []
for element in tqdm(dataset, total=len(dataset)):
    text = element[0]
    temp_outputs = []
    for function in element[1]["functions"]:
        function_name = function["function_name"]
        kwargs = {}
        inputs = function["inputs"]
        if len(inputs):
            for inp in inputs:
                name, value = format_value_new(function_name=function_name, inputs=inp)
                kwargs[name] = value

        temp_outputs.append({"function": function_name, "kwargs": kwargs})
    outputs.append({"input": text, "output": temp_outputs})

  0%|          | 0/986 [00:00<?, ?it/s]

## Push to HF

In [7]:
df = pd.DataFrame(data=outputs)
print(f"Number of data samples: {len(df)}")

Number of data samples: 986


In [8]:
df.head()

,input,output
0,"Move robot TCP to coordinates (0.5, 0.3, 0.7) ...","[{'function': 'move_tcp', 'kwargs': {'x': 500...."
1,Rotate the 6th joint by -30 degrees,"[{'function': 'move_joint', 'kwargs': {'joint'..."
2,Provide me with the current status of robot jo...,"[{'function': 'get_joint_values', 'kwargs': {}}]"
3,Rotate the robot base by 45 degrees and move t...,"[{'function': 'move_joint', 'kwargs': {'joint'..."
4,"Please rotate joint 2 by 30 degrees, joint 7 b...","[{'function': 'move_joint', 'kwargs': {'joint'..."


In [9]:
dataset = Dataset.from_pandas(df)

Create test and train split <br>
In the future we should make sure that all "functions" are represented equally

In [10]:
train_test_split = dataset.train_test_split(test_size=0.1)

In [11]:
repo_id = "Studeni/robot-instructions"

In [12]:
train_test_split["train"].push_to_hub(
    repo_id=repo_id, split="train", token=os.environ["HF_TOKEN"]
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Studeni/robot-instructions/commit/397788498bb675b2246e4b0fb013ecced5b05051', commit_message='Upload dataset', commit_description='', oid='397788498bb675b2246e4b0fb013ecced5b05051', pr_url=None, pr_revision=None, pr_num=None)

In [13]:
train_test_split["test"].push_to_hub(
    repo_id=repo_id, split="test", token=os.environ["HF_TOKEN"]
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/760 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Studeni/robot-instructions/commit/8aabc678a6837951964ac30197e20bfb287ab844', commit_message='Upload dataset', commit_description='', oid='8aabc678a6837951964ac30197e20bfb287ab844', pr_url=None, pr_revision=None, pr_num=None)